In [7]:
# 1. Replace print statements with logging (INFO, WARNING, ERROR).
# Logging is implemented inside etl_pipeline.py.
# INFO = normal progress, WARNING = optional input missing, ERROR = processing failure.

from etl_pipeline import run_pipeline
import logging

print("Logging functions are ready in etl_pipeline.py")

Logging functions are ready in etl_pipeline.py


In [ ]:
# 2. Write a log file per run with a timestamp.

from pathlib import Path

Path("logs").mkdir(exist_ok=True)
print("Log folder is ready:", Path("logs").resolve())

Log folder is ready: C:\Users\Niranjan\Documents\sample\day14_olist_sample_style\logs


In [14]:
# 3. Move paths, thresholds and mappings into config.yaml.

import yaml

with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("Input folder:", config["input_dir"])
print("Output folder:", config["output_dir"])
print("Rejection threshold:", config["max_rejection_percent"])
print("Datasets:", list(config["datasets"].keys()))

Input folder: ..
Output folder: output
Rejection threshold: 20.0
Datasets: ['customers', 'products', 'geolocation', 'sellers', 'order_items', 'order_payments', 'product_category_translation', 'order_reviews', 'orders']


In [10]:
# 4. Write the clean output as CSV and JSON.

from pathlib import Path
import pandas as pd
import time
from etl_pipeline import load_config, validate_rows

task4_start_time = time.perf_counter()

config = load_config("config.yaml")
input_dir = Path(config["input_dir"])
output_dir = Path(config["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

orders_rules = config["datasets"]["orders"]
orders_path = input_dir / orders_rules["file"]

orders_df = pd.read_csv(
    orders_path,
    dtype=str,
    keep_default_na=False
)

clean_orders, rejected_orders = validate_rows(
    orders_df,
    "orders",
    orders_rules
)

clean_orders.to_csv(
    output_dir / "orders_clean.csv",
    index=False
)

clean_orders.to_json(
    output_dir / "orders_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)

if not rejected_orders.empty:
    rejected_orders.to_csv(
        output_dir / "orders_bad_records.csv",
        index=False
    )

print("Clean CSV and JSON files saved.")
print("Valid rows:", len(clean_orders))
print("Rejected rows:", len(rejected_orders))

Clean CSV and JSON files saved.
Valid rows: 99441
Rejected rows: 0


In [11]:
# 5. Write a run summary JSON (counts, duration).

import json
from datetime import datetime
from pathlib import Path

task4_duration = round(time.perf_counter() - task4_start_time, 3)

total_rows = len(orders_df)
valid_rows = len(clean_orders)
rejected_rows = len(rejected_orders)

run_summary = {
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "dataset": "orders",
    "total_rows": total_rows,
    "valid_rows": valid_rows,
    "rejected_rows": rejected_rows,
    "duration_seconds": task4_duration
}

summary_dir = output_dir / "summaries"
summary_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = summary_dir / f"orders_run_summary_{timestamp}.json"

with summary_path.open("w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=4)

print("Run summary saved:", summary_path)
print(run_summary)

Run summary saved: output\summaries\orders_run_summary_20261125_103717.json
{'run_timestamp': '2026-11-25T10:37:17', 'dataset': 'orders', 'total_rows': 99441, 'valid_rows': 99441, 'rejected_rows': 0, 'duration_seconds': 248.257}


In [12]:
# 6. Keep a manifest of processed files and skip them next time.

import json
from pathlib import Path

manifest_path = Path(config["manifest_path"])
if manifest_path.exists():
    print(json.loads(manifest_path.read_text(encoding="utf-8")))
else:
    print("Manifest will be created after the first successful pipeline run.")

Manifest will be created after the first successful pipeline run.


In [22]:
# 7. Run the whole pipeline from one notebook cell using config.yaml.

from etl_pipeline import run_pipeline

first_run = run_pipeline("config.yaml")

print("Pipeline Summary:")
print(first_run)

2026-11-25 10:58:55,878 | INFO | Starting ETL run 20261125T052855Z
2026-11-25 10:58:55,896 | INFO | Skipping unchanged file: customers
2026-11-25 10:58:55,903 | INFO | Skipping unchanged file: products
2026-11-25 10:58:55,989 | INFO | Skipping unchanged file: geolocation
2026-11-25 10:58:55,991 | INFO | Skipping unchanged file: sellers
2026-11-25 10:58:56,011 | INFO | Skipping unchanged file: order_items
2026-11-25 10:58:56,019 | INFO | Skipping unchanged file: order_payments
2026-11-25 10:58:56,021 | INFO | Skipping unchanged file: product_category_translation
2026-11-25 10:58:56,043 | INFO | Skipping unchanged file: order_reviews
2026-11-25 10:58:56,066 | INFO | Skipping unchanged file: orders
2026-11-25 10:58:56,068 | INFO | Finished run: processed=0 skipped=9 failed=0 duration=0.200s


Pipeline Summary:
{'run_id': '20261125T052855Z', 'duration_seconds': 0.2, 'processed_count': 0, 'skipped_count': 9, 'failed_count': 0, 'processed': [], 'skipped': ['customers', 'products', 'geolocation', 'sellers', 'order_items', 'order_payments', 'product_category_translation', 'order_reviews', 'orders'], 'failed': [], 'log_file': 'logs\\etl_run_20261125_105855_877333.log'}


In [26]:
# 8. Lint your .py modules with ruff (or flake8) and fix the warnings.

import subprocess
import sys

lint = subprocess.run(
    [sys.executable, "-m", "ruff", "check", "etl_pipeline.py"],
    capture_output=True,
    text=True
)
print(lint.stdout)
print(lint.stderr)
print("Ruff exit code:", lint.returncode)

fix = subprocess.run(
    [sys.executable, "-m", "ruff", "check", "--fix", "etl_pipeline.py"],
    capture_output=True,
    text=True
)
print(fix.stdout)
print(fix.stderr)

lint_again = subprocess.run(
    [sys.executable, "-m", "ruff", "check", "etl_pipeline.py"],
    capture_output=True,
    text=True
)
print(lint_again.stdout)
print(lint_again.stderr)
print("Final Ruff exit code:", lint_again.returncode)

All checks passed!


Ruff exit code: 0
All checks passed!


All checks passed!


Final Ruff exit code: 0


In [ ]:
# 9. Address review comments and merge.
# Run these commands in a terminal from your Git project folder.
#
# git status
# git switch -c day14-olist-etl
# git add day14_olist_etl.ipynb config.yaml etl_pipeline.py requirements.txt
# git commit -m "Add configurable Olist ETL pipeline"
# git push -u origin day14-olist-etl
#
# Open a Pull Request on GitHub, address reviewer comments,
# push the fixes, then merge after approval.

In [17]:
# 10. Re-run the notebook and show that the second run processes nothing new.
# Do not modify the source CSV files or config.yaml between these runs.

second_run = run_pipeline("config.yaml")
second_run

assert second_run["processed_count"] == 0, "Some files were processed again."
assert second_run["failed_count"] == 0, "Check the run summary and log for failures."

print("Second run check passed: no unchanged files were processed.")

2026-11-25 10:48:14,614 | INFO | Starting ETL run 20261125T051814Z
2026-11-25 10:48:14,638 | INFO | Skipping unchanged file: customers
2026-11-25 10:48:14,645 | INFO | Skipping unchanged file: products
2026-11-25 10:48:14,724 | INFO | Skipping unchanged file: geolocation
2026-11-25 10:48:14,726 | INFO | Skipping unchanged file: sellers
2026-11-25 10:48:14,746 | INFO | Skipping unchanged file: order_items
2026-11-25 10:48:14,756 | INFO | Skipping unchanged file: order_payments
2026-11-25 10:48:14,758 | INFO | Skipping unchanged file: product_category_translation
2026-11-25 10:48:14,780 | INFO | Skipping unchanged file: order_reviews
2026-11-25 10:48:14,809 | INFO | Skipping unchanged file: orders
2026-11-25 10:48:14,811 | INFO | Finished run: processed=0 skipped=9 failed=0 duration=0.217s


Second run check passed: no unchanged files were processed.


In [24]:
!ruff check etl_pipeline.py

TRY004 Prefer `TypeError` exception for invalid type
  --> etl_pipeline.py:35:9
   |
33 |         config = yaml.safe_load(file)
34 |     if not isinstance(config, dict):
35 |         raise ValueError("config.yaml must contain a YAML mapping.")
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
36 |     return config
   |

Found 1 error.


In [25]:
!ruff check etl_pipeline.py

All checks passed!
